# Hi AI Clip Worker (Kaggle GPU)
Reads `job.json` (from GH Actions dataset or /kaggle/input), generates short 5s clips, writes to `/kaggle/working/clips/`, then exits so GPU quota stops.

In [ ]:
import json, os, time, pathlib, subprocess, sys
WORK = pathlib.Path('/kaggle/working')
CLIPS = WORK / 'clips'
CLIPS.mkdir(parents=True, exist_ok=True)

def find_job():
    candidates = [
        WORK / 'job.json',
        pathlib.Path('/kaggle/input') ,
    ]
    p = WORK / 'job.json'
    if p.exists():
        return json.loads(p.read_text())
    # search datasets
    root = pathlib.Path('/kaggle/input')
    if root.exists():
        for f in root.rglob('job.json'):
            return json.loads(f.read_text())
    # default dry job
    return {
        'job_id': 'dry-' + str(int(time.time())),
        'channel': 'teen',
        'clip_seconds': 5,
        'num_clips': 2,
        'prompts': [
            'cinematic vertical shot of a student studying at night, soft light, no text',
            'calm atmosphere city window rain, vertical, no text'
        ]
    }

job = find_job()
print('JOB', job.get('job_id'), 'clips', job.get('num_clips'))
(WORK / 'job_echo.json').write_text(json.dumps(job, indent=2))


In [ ]:
# Lightweight placeholder: generate solid-color mp4s so the GH→Kaggle→GH loop is testable
# Replace this cell later with LTX-Video / Wan 1.3B inference.
import shutil
sec = float(job.get('clip_seconds') or 5)
n = int(job.get('num_clips') or 2)
prompts = job.get('prompts') or ['placeholder']
manifest = []
for i in range(n):
    out = CLIPS / f'clip_{i:02d}.mp4'
    color = ['black','gray','navy','maroon'][i % 4]
    cmd = [
        'ffmpeg','-y','-f','lavfi','-i',f'color=c={color}:s=1080x1920:d={sec}',
        '-c:v','libx264','-pix_fmt','yuv420p','-t',str(sec),str(out)
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        ok = out.exists()
    except Exception as e:
        print('ffmpeg fail', e)
        ok = False
    manifest.append({
        'index': i,
        'path': str(out.name),
        'prompt': prompts[i % len(prompts)],
        'ok': ok,
        'seconds': sec
    })
    print('clip', i, ok)

(WORK / 'manifest.json').write_text(json.dumps({
    'job_id': job.get('job_id'),
    'channel': job.get('channel'),
    'clips': manifest,
    'status': 'complete'
}, indent=2))
print('DONE — kernel will exit and free GPU')
